In [1]:
# ============================================================
# Cell 1 — Imports and setup
# ============================================================
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

RAW_PATH = '../data/raw/mental-health-in-tech-2016.csv'
INVENTORY_PATH = '../data/processed/column_inventory.csv'
ROUTING_PATH = '../data/processed/routing_flags.csv'

In [2]:
# ============================================================
# Cell 2 — Load raw data and the column inventory from Notebook 1
# ============================================================
df = pd.read_csv(RAW_PATH)
inventory = pd.read_csv(INVENTORY_PATH)
routing_flags = pd.read_csv(ROUTING_PATH)

print(f"Raw data shape: {df.shape}")
print(f"Inventory rows: {inventory.shape[0]}")

inventory.head(5)

Raw data shape: (1433, 63)
Inventory rows: 63


,short_id,column_number,question_text,question_short,block,dtype,n_missing,pct_missing,n_unique,top_category,top_category_pct,missingness_cluster,missingness_cluster_k4,role
0,q01,0,Are you self-employed?,Are you self-employed?...,A_employment_context,int64,0,0.00,2,0,79.97,3,4,context
1,q02,1,How many employees does your company or organi...,How many employees does your company or organi...,A_employment_context,str,287,20.03,6,26-100,25.48,2,3,context
2,q03,2,Is your employer primarily a tech company/orga...,Is your employer primarily a tech company/orga...,A_employment_context,float64,287,20.03,2,1.0,77.05,2,3,context
3,q04,3,Is your primary role within your company relat...,Is your primary role within your company relat...,A_employment_context,float64,1170,81.65,2,1.0,94.30,1,2,near_constant
4,q05,4,Does your employer provide mental health benef...,Does your employer provide mental health benef...,B_current_employer,str,287,20.03,4,Yes,46.34,2,3,clustering_candidate


In [3]:
# ============================================================
# Cell 3 — Rename columns to short IDs using POSITION, not exact
# string matching. Exact-string matching against question_text is
# fragile (whitespace, smart quotes, minor text edits between
# files) and silently fails without raising an error — it just
# leaves columns unrenamed. Position-based renaming via
# column_number is robust to all of that, since raw CSV column
# order is guaranteed stable.
# ============================================================
inventory_sorted = inventory.sort_values('column_number').reset_index(drop=True)

assert len(inventory_sorted) == df.shape[1], (
    f"Inventory has {len(inventory_sorted)} rows but df has {df.shape[1]} columns "
    "— counts must match before renaming by position."
)

position_rename_map = dict(zip(df.columns, inventory_sorted['short_id']))
df = df.rename(columns=position_rename_map)

question_lookup = dict(zip(inventory_sorted['short_id'], inventory_sorted['question_short']))

print(df.columns[:10].tolist())
print("\nDoes df contain 'q01' now?", 'q01' in df.columns)

['q01', 'q02', 'q03', 'q04', 'q05', 'q06', 'q07', 'q08', 'q09', 'q10']

Does df contain 'q01' now? True


In [4]:
# ============================================================
# Cell 3b — Sanity check the rename against known facts from EDA
# q01 (self-employed) should be binary 0/1, and q56 (age) should
# be numeric — if these checks fail, the column order assumption
# is wrong and needs manual inspection.
# ============================================================
print("q01 unique values (expect 0/1):", df['q01'].unique())
print("q56 dtype (expect numeric):", df['q56'].dtype)
print("q56 sample values:", df['q56'].head().tolist())

q01 unique values (expect 0/1): [0 1]
q56 dtype (expect numeric): int64
q56 sample values: [39, 29, 38, 43, 43]


In [5]:
# ============================================================
# Cell 3c — Confirm whether self-employed filtering has run yet
# ============================================================
print(f"Current row count: {df.shape[0]}")
print("Expected 1146 if Cell 4 already ran successfully, 1433 if not yet")

Current row count: 1433
Expected 1146 if Cell 4 already ran successfully, 1433 if not yet


In [6]:
# ============================================================
# Cell 4 — Filter out self-employed respondents
# EDA (Section 9) established that Block B is 100% missing for the
# self-employed and Block C is 100% missing for the employed —
# this is a structural fork in the survey, not random noise.
# The case study targets company employees, so we drop q01 == 1.
# ============================================================
n_before = len(df)
df = df[df['q01'] == 0].copy()   # 0 = not self-employed
n_after = len(df)

print(f"Filtered self-employed respondents: {n_before} -> {n_after} rows "
      f"({n_before - n_after} removed)")

Filtered self-employed respondents: 1433 -> 1146 rows (287 removed)


In [7]:
# ============================================================
# Cell 5 — Drop the now-irrelevant Block C columns and the q01 flag
# Block C (q17-q24) only ever applied to self-employed respondents,
# who have just been removed, so the block carries no more information.
# ============================================================
block_c_cols = inventory.loc[
    inventory['block'] == 'C_self_employed', 'short_id'
].tolist()

df = df.drop(columns=block_c_cols + ['q01'])

print(f"Dropped {len(block_c_cols) + 1} columns. New shape: {df.shape}")

Dropped 9 columns. New shape: (1146, 54)


In [8]:
# ============================================================
# Cell 6 — Clean implausible age values
# EDA (Section 6.2) flagged ages > 100 as data entry errors.
# We also guard the lower bound, since ages < 18 are implausible
# for a workplace survey. Implausible values are set to NaN first
# so they don't distort the median used for imputation below.
# ============================================================
print("Age summary before cleaning:")
print(df['q56'].describe())

n_flagged = df['q56'][(df['q56'] < 18) | (df['q56'] > 100)].shape[0]
df.loc[(df['q56'] < 18) | (df['q56'] > 100), 'q56'] = np.nan

print(f"\nFlagged and nulled {n_flagged} implausible age values")

Age summary before cleaning:
count    1146.000000
mean       33.655323
std        11.703277
min         3.000000
25%        28.000000
50%        32.000000
75%        38.000000
max       323.000000
Name: q56, dtype: float64

Flagged and nulled 3 implausible age values


In [9]:
# ============================================================
# Cell 7 — Impute cleaned age with the median
# EDA found only a weak relationship between age and the stigma /
# disclosure constructs (near-identical medians across categories),
# so a simple median fill is a defensible, low-risk choice here.
# ============================================================
median_age = df['q56'].median()
df['q56'] = df['q56'].fillna(median_age)

print(f"Imputed missing/cleaned ages with median: {median_age}")

Imputed missing/cleaned ages with median: 32.0


In [10]:
# ============================================================
# Cell 8 — Encode structural "Not applicable" for Block D
# EDA proved Block D missingness is 100% deterministic on q25
# (has previous employers). This is NOT missing data to impute —
# it is a valid third category meaning "this question did not apply".
# Imputing it with mean/mode would fabricate an opinion the
# respondent never had the chance to give.
# ============================================================
block_d_cols = inventory.loc[
    inventory['block'] == 'D_previous_employer', 'short_id'
].tolist()
block_d_cols = [c for c in block_d_cols if c != 'q25']  # keep the flag itself untouched

for col in block_d_cols:
    df[col] = df[col].fillna('Not applicable')

print(f"Encoded 'Not applicable' for {len(block_d_cols)} Block D columns")

Encoded 'Not applicable' for 11 Block D columns


In [11]:
# ============================================================
# Cell 9 — Encode structural "Not applicable" for the four smaller
# gated question groups identified in the routing_flags artifact
# (q44->q45, q47->q49/q50, q51->q52, country->q59/q61).
# Same logic as Cell 8: these are skip patterns, not random gaps.
# ============================================================
gated_by_flag = {
    'q45': 'q44',   # observed unsupportive response -> follow-up
    'q49': 'q47',   # currently has MH disorder -> diagnosis list
    'q50': 'q47',   # currently has MH disorder -> suspected conditions
    'q52': 'q51',   # diagnosed by professional -> diagnosis list
}

for col in gated_by_flag:
    df[col] = df[col].fillna('Not applicable')

# q59 (state of residence) and q61 (state of work) only apply to
# respondents living/working in the United States
df['q59'] = df['q59'].fillna('Not applicable')
df['q61'] = df['q61'].fillna('Not applicable')

print("Encoded 'Not applicable' for gated follow-up questions and US-state fields")

Encoded 'Not applicable' for gated follow-up questions and US-state fields


In [12]:
# ============================================================
# Cell 9b — Resolve q06 and q44 residual missingness
#
# q06: gated by q05, though not with a single deterministic
# value — missingness rises sharply as q05 answers become less
# certain or less applicable ('Yes' 0% -> "I don't know" 1.9% ->
# 'No' 33.8% -> 'Not eligible / N/A' 66.3%). This is a seventh
# routing pattern EDA's original six documented flags (Section
# 4.5) did not catch. Treated as structurally not-applicable for
# respondents whose q05 answer was 'No' or 'Not eligible for
# coverage / N/A', matching the same convention used for every
# other routing-gated column in Cells 8-9. The residual small
# gaps within 'Yes'/"I don't know" respondents are true item
# non-response and handled with the fallback below.
#
# q44: no gating pattern found against q43 or any other nearby
# column - genuine item non-response, not a structural skip.
# Filled with the column mode. At 52/1146 (4.5%) missing, this
# is a low-risk, documented simplification, consistent with the
# reasoning already used for age (Cell 7).
# ============================================================

q05_na_trigger = df['q05'].isin(['No', 'Not eligible for coverage / N/A'])
df.loc[q05_na_trigger & df['q06'].isna(), 'q06'] = 'Not applicable'

remaining_q06_na = df['q06'].isna().sum()
if remaining_q06_na > 0:
    q06_mode = df['q06'].mode()[0]
    df['q06'] = df['q06'].fillna(q06_mode)
    print(f"q06: filled {q05_na_trigger.sum()} rows as 'Not applicable' via q05 gate, "
          f"{remaining_q06_na} residual rows via mode ('{q06_mode}')")
else:
    print(f"q06: filled {q05_na_trigger.sum()} rows as 'Not applicable' via q05 gate, no residual gaps")

q44_mode = df['q44'].mode()[0]
n_q44_missing = df['q44'].isna().sum()
df['q44'] = df['q44'].fillna(q44_mode)
print(f"q44: filled {n_q44_missing} rows via mode ('{q44_mode}')")

q06: filled 296 rows as 'Not applicable' via q05 gate, 6 residual rows via mode ('No')
q44: filled 52 rows via mode ('No')


In [13]:
# ============================================================
# Cell 10 — Inspect raw category values for the six columns flagged
# as possible Likert items. This is a read-only inspection step —
# no mapping is applied yet. The actual standardization happens
# later (Cell 19), after we've grouped ALL 36 clustering columns
# by their real vocabulary (Cell 18), not just these six assumed
# up front. This cell is kept as an early spot-check.
# ============================================================
likert_candidate_cols = ['q30', 'q31', 'q32', 'q33', 'q41', 'q42']

for col in likert_candidate_cols:
    print(f"\n{col}: {sorted(df[col].dropna().unique(), key=str)}")


q30: ["I don't know", 'No', 'Not applicable', 'Sometimes', 'Yes, always']

q31: ["I don't know", 'None of them', 'Not applicable', 'Some of them', 'Yes, all of them']

q32: ['None of them', 'Not applicable', 'Some of them', 'Yes, all of them']

q33: ['No, at none of my previous employers', 'Not applicable', 'Some of my previous employers', 'Yes, at all of my previous employers']

q41: ['Maybe', "No, I don't think it would", 'No, it has not', 'Yes, I think it would', 'Yes, it has']

q42: ['Maybe', "No, I don't think they would", 'No, they do not', 'Yes, I think they would', 'Yes, they do']


In [14]:
# ============================================================
# Cell 11 — Consolidate gender into a small, consistent set
# Raw gender has 70 unique values (spelling/case variants).
# EDA flagged this as too noisy for direct use. We normalize
# to lowercase, strip whitespace, and bucket into five categories.
# ============================================================

def consolidate_gender(val):
    if pd.isna(val):
        return 'Prefer not to say'
    v = str(val).strip().lower()
    if v in {'male', 'm', 'man', 'cis male', 'male.', 'cis man'}:
        return 'Male'
    if v in {'female', 'f', 'woman', 'cis female', 'cis-woman'}:
        return 'Female'
    if 'non' in v or 'enby' in v or 'nb' in v:
        return 'Non-binary'
    return 'Other'

df['q57'] = df['q57'].apply(consolidate_gender)

print(df['q57'].value_counts())

q57
Male                 840
Female               261
Other                 34
Non-binary             8
Prefer not to say      3
Name: count, dtype: int64


In [15]:
# ============================================================
# Cell 12 — Consolidate country into top categories + "Other"
# EDA found the dataset is 58.6% US respondents with a long tail
# of other countries. Keeping all ~53 raw values would make
# country a high-cardinality nuisance feature; we keep only the
# top markets identified in EDA and bucket the rest.
# ============================================================
top_countries = {'United States of America', 'United Kingdom', 'Canada', 'Germany'}

df['q58'] = df['q58'].apply(lambda x: x if x in top_countries else 'Other')

print(df['q58'].value_counts())

q58
United States of America    709
Other                       196
United Kingdom              134
Canada                       63
Germany                      44
Name: count, dtype: int64


In [16]:
# ============================================================
# Cell 13 — Consolidate free-text diagnosis fields
# q49, q50, q52 contain 99-128 unique free-text values each.
# We keep the most frequent conditions found in EDA's word
# analysis and bucket everything else into "Other". Rows already
# holding "Not applicable" (Cell 9) are passed through unchanged.
# ============================================================

diagnosis_cols = ['q49', 'q50', 'q52']
top_conditions = ['Anxiety Disorder', 'Mood Disorder', 'Depression', 'Attention Deficit']

def consolidate_diagnosis(text):
    if pd.isna(text) or text == 'Not applicable':
        return 'Not applicable'
    text_lower = str(text).lower()
    for condition in top_conditions:
        if condition.lower() in text_lower:
            return condition
    return 'Other'

for col in diagnosis_cols:
    df[col + '_consolidated'] = df[col].apply(consolidate_diagnosis)

print(df[[c + '_consolidated' for c in diagnosis_cols]].apply(pd.Series.value_counts))

                   q49_consolidated  q50_consolidated  q52_consolidated
Not applicable                  701               896               583
Anxiety Disorder                272               149               295
Mood Disorder                   128                72               210
Other                            26                21                32
Attention Deficit                19                 8                26


In [17]:
# ============================================================
# Cell 14 — Re-check missingness and near-constant status
# post-filtering. Removing 287 rows shifts every percentage
# computed in Notebook 1, and may surface NEW near-constant
# columns or resolve previously flagged ones. Open question #6
# from EDA requires this re-check before roles are finalized.
# ============================================================

def summarize_column(s):
    n_missing = s.isna().sum()
    n_unique = s.nunique(dropna=True)
    top_pct = (
        s.value_counts(normalize=True).iloc[0] * 100
        if n_unique > 0 else np.nan
    )
    return pd.Series({
        'n_missing': n_missing,
        'pct_missing': n_missing / len(s) * 100,
        'n_unique': n_unique,
        'top_category_pct': top_pct
    })

post_filter_summary = df.apply(summarize_column).T
near_constant_now = post_filter_summary[post_filter_summary['top_category_pct'] >= 90]

print("Columns that are now near-constant (>=90% single category):")
print(near_constant_now)



Columns that are now near-constant (>=90% single category):
     n_missing  pct_missing  n_unique  top_category_pct
q04      883.0    77.050611       2.0         94.296578
q16        0.0     0.000000       2.0         91.448517


In [18]:
# ============================================================
# Cell 15 — Drop columns confirmed near-constant after review
#
# Cell 14 flagged two columns: q04 and q16.
#   - q04 (primary role related to tech/IT): near-constant AND
#     77% missing, but it was already slated for the 'context'
#     role (not clustering input), so it stays in df for
#     post-hoc cluster profiling rather than being dropped outright.
#   - q16 (observed negative consequences for co-workers): 91.4%
#     one category, but the 8.6% minority represents respondents
#     who witnessed real workplace stigma events — arguably one
#     of the most HR-actionable signals in the dataset. Kept as
#     a clustering candidate, not dropped.
#
# Conclusion: no columns dropped at this step. This is a
# deliberate outcome of the review, not a skipped check.
# Re-confirm this decision still holds now that q30-q33/q41/q42
# are no longer corrupted — check Cell 14's output above for any
# of those six appearing near the 90% threshold.
# ============================================================
columns_to_drop = []  # confirmed empty after reviewing Cell 14 output — see rationale above

df = df.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} confirmed near-constant columns")
print(f"Shape after drop: {df.shape}")

Dropped 0 confirmed near-constant columns
Shape after drop: (1146, 57)


In [19]:
# ============================================================
# Cell 15a — Guard rail: confirm q16 wasn't accidentally dropped
# ============================================================
assert 'q16' in df.columns, "q16 was dropped but is needed as a clustering candidate"
assert 'q04' in df.columns, "q04 was dropped but is needed as a context variable"
print("Guard rail passed: q04 and q16 both retained in df")

Guard rail passed: q04 and q16 both retained in df


In [20]:
# ============================================================
# Cell 16 — Split into the clustering matrix and the context frame
# Clustering candidates (Blocks B, D, F, G, H per EDA role
# assignment) go into X_cat. Demographics and the previous-employer
# routing flag (q25) are kept separately for post-hoc cluster
# profiling only, per EDA Section 10's rationale for excluding
# demographics from the clustering inputs themselves.
# ============================================================
clustering_cols = inventory.loc[
    inventory['role'] == 'clustering_candidate', 'short_id'
].tolist()
clustering_cols = [c for c in clustering_cols if c in df.columns]

context_cols = ['q56', 'q57', 'q58', 'q62', 'q63', 'q25']
context_cols = [c for c in context_cols if c in df.columns]

X_cat = df[clustering_cols].copy()
context = df[context_cols].copy()

print(f"Clustering matrix: {X_cat.shape}")
print(f"Context frame: {context.shape}")

Clustering matrix: (1146, 36)
Context frame: (1146, 6)


In [21]:
# ============================================================
# Cell 16a — Final guard rail (confirms q25's role label directly,
# not just the resulting count — catches both the "q25 in
# clustering_candidate" bug and the Cell 11-removal fix at once)
# ============================================================
assert 'q25' not in X_cat.columns
assert 'q25' in context.columns
assert X_cat.shape[1] == 36
print("Guard rail passed.")


Guard rail passed.


In [22]:
# ============================================================
# Cell 17 — Inspect raw vocabulary for every clustering column
# This is the FIRST time all 36 columns are inspected together,
# with clean data (no prior corruption from a premature mapping
# step). This inspection drives every encoding decision that follows.
# ============================================================
for c in sorted(X_cat.columns):
    vals = sorted(X_cat[c].dropna().unique(), key=lambda x: str(x))
    print(f"{c:>4} ({len(vals):>2} values): {vals}")

 q05 ( 4 values): ["I don't know", 'No', 'Not eligible for coverage / N/A', 'Yes']
 q06 ( 4 values): ['I am not sure', 'No', 'Not applicable', 'Yes']
 q07 ( 3 values): ["I don't know", 'No', 'Yes']
 q08 ( 3 values): ["I don't know", 'No', 'Yes']
 q09 ( 3 values): ["I don't know", 'No', 'Yes']
 q10 ( 6 values): ["I don't know", 'Neither easy nor difficult', 'Somewhat difficult', 'Somewhat easy', 'Very difficult', 'Very easy']
 q11 ( 3 values): ['Maybe', 'No', 'Yes']
 q12 ( 3 values): ['Maybe', 'No', 'Yes']
 q13 ( 3 values): ['Maybe', 'No', 'Yes']
 q14 ( 3 values): ['Maybe', 'No', 'Yes']
 q15 ( 3 values): ["I don't know", 'No', 'Yes']
 q26 ( 5 values): ["I don't know", 'No, none did', 'Not applicable', 'Some did', 'Yes, they all did']
 q27 ( 5 values): ['I was aware of some', 'N/A (not currently aware)', 'No, I only became aware later', 'Not applicable', 'Yes, I was aware of all of them']
 q28 ( 5 values): ["I don't know", 'None did', 'Not applicable', 'Some did', 'Yes, they all did']
 q

In [23]:
# ============================================================
# Cell 18 — Group columns by their distinct vocabulary
# Buckets columns that share an identical set of raw category
# strings, so one mapping dict can serve every column in that
# bucket, and outlier columns whose wording doesn't match their
# block's expected pattern are easy to spot.
# ============================================================
from collections import defaultdict

vocab_groups = defaultdict(list)
for c in X_cat.columns:
    vocab = tuple(sorted(X_cat[c].dropna().unique(), key=lambda x: str(x)))
    vocab_groups[vocab].append(c)

for vocab, cols in vocab_groups.items():
    print(f"\nVocabulary: {vocab}")
    print(f"Columns ({len(cols)}): {sorted(cols)}")


Vocabulary: ("I don't know", 'No', 'Not eligible for coverage / N/A', 'Yes')
Columns (1): ['q05']

Vocabulary: ('I am not sure', 'No', 'Not applicable', 'Yes')
Columns (1): ['q06']

Vocabulary: ("I don't know", 'No', 'Yes')
Columns (5): ['q07', 'q08', 'q09', 'q15', 'q46']

Vocabulary: ("I don't know", 'Neither easy nor difficult', 'Somewhat difficult', 'Somewhat easy', 'Very difficult', 'Very easy')
Columns (1): ['q10']

Vocabulary: ('Maybe', 'No', 'Yes')
Columns (8): ['q11', 'q12', 'q13', 'q14', 'q37', 'q39', 'q47', 'q48']

Vocabulary: ("I don't know", 'No, none did', 'Not applicable', 'Some did', 'Yes, they all did')
Columns (1): ['q26']

Vocabulary: ('I was aware of some', 'N/A (not currently aware)', 'No, I only became aware later', 'Not applicable', 'Yes, I was aware of all of them')
Columns (1): ['q27']

Vocabulary: ("I don't know", 'None did', 'Not applicable', 'Some did', 'Yes, they all did')
Columns (2): ['q28', 'q35']

Vocabulary: ('None did', 'Not applicable', 'Some did', '

In [24]:

# ============================================================
# Cell 19 — Build per-vocabulary ordinal encoding maps and apply
# them to X_cat.
#
# Encoding rule (applied consistently across all groups):
#   -1  = structural "not applicable" / N/A response
#    0  = most negative / least frequent / least open substantive answer
#  1..n = ascending degree, ending at most positive / most frequent /
#         most open substantive answer
#   uncertain answers ("Maybe", "I don't know", "I am not sure") sit
#   as a middle code between "no" and "yes", EXCEPT in q10 where
#   "I don't know" doesn't fit the ease-difficulty spectrum at all
#   and is treated as -1 alongside true N/A.
#
# q41/q42 special case: these mix two respondent populations
# (experienced: has/has not vs. hypothetical: would/wouldn't).
# The two "no" variants are collapsed to one code, and the two
# "yes" variants to another, rather than kept as 5 separate
# ordinal steps — a deliberate simplification, not a neutral
# technical default. Document this in the write-up.
#
# q06 special case: after Cell 9b filled 296 structurally-gated
# rows with 'Not applicable', q06's vocabulary grew from 3 values
# to 4. Its key below is written in the exact alphabetical order
# vocab_groups produces ('No' sorts before 'Not applicable' since
# 'No' is a string-prefix of it) — an earlier version of this
# dict had the wrong order and silently failed to match.
#
# q53 is excluded here — already numeric 0/1, left untouched.
# ============================================================

vocab_maps = {
    ("I don't know", 'No', 'Not eligible for coverage / N/A', 'Yes'): {
        'No': 0, "I don't know": 1, 'Yes': 2,
        'Not eligible for coverage / N/A': -1
    },
    ('I am not sure', 'No', 'Not applicable', 'Yes'): {
        'No': 0, 'I am not sure': 1, 'Yes': 2, 'Not applicable': -1
    },
    ("I don't know", 'No', 'Yes'): {
        'No': 0, "I don't know": 1, 'Yes': 2
    },
    ("I don't know", 'Neither easy nor difficult', 'Somewhat difficult',
     'Somewhat easy', 'Very difficult', 'Very easy'): {
        'Very easy': 0, 'Somewhat easy': 1, 'Neither easy nor difficult': 2,
        'Somewhat difficult': 3, 'Very difficult': 4, "I don't know": -1
    },
    ('Maybe', 'No', 'Yes'): {
        'No': 0, 'Maybe': 1, 'Yes': 2
    },
    ("I don't know", 'No, none did', 'Not applicable', 'Some did', 'Yes, they all did'): {
        'No, none did': 0, "I don't know": 1, 'Some did': 2,
        'Yes, they all did': 3, 'Not applicable': -1
    },
    ('I was aware of some', 'N/A (not currently aware)', 'No, I only became aware later',
     'Not applicable', 'Yes, I was aware of all of them'): {
        'No, I only became aware later': 0, 'I was aware of some': 1,
        'Yes, I was aware of all of them': 2,
        'N/A (not currently aware)': -1, 'Not applicable': -1
    },
    ("I don't know", 'None did', 'Not applicable', 'Some did', 'Yes, they all did'): {
        'None did': 0, "I don't know": 1, 'Some did': 2,
        'Yes, they all did': 3, 'Not applicable': -1
    },
    ('None did', 'Not applicable', 'Some did', 'Yes, they all did'): {
        'None did': 0, 'Some did': 1, 'Yes, they all did': 2, 'Not applicable': -1
    },
    ("I don't know", 'No', 'Not applicable', 'Sometimes', 'Yes, always'): {
        'No': 0, "I don't know": 1, 'Sometimes': 2, 'Yes, always': 3, 'Not applicable': -1
    },
    ("I don't know", 'None of them', 'Not applicable', 'Some of them', 'Yes, all of them'): {
        'None of them': 0, "I don't know": 1, 'Some of them': 2,
        'Yes, all of them': 3, 'Not applicable': -1
    },
    ('None of them', 'Not applicable', 'Some of them', 'Yes, all of them'): {
        'None of them': 0, 'Some of them': 1, 'Yes, all of them': 2, 'Not applicable': -1
    },
    ('No, at none of my previous employers', 'Not applicable',
     'Some of my previous employers', 'Yes, at all of my previous employers'): {
        'No, at none of my previous employers': 0, 'Some of my previous employers': 1,
        'Yes, at all of my previous employers': 2, 'Not applicable': -1
    },
    ("I don't know", 'No, at none of my previous employers', 'Not applicable',
     'Some of my previous employers', 'Yes, at all of my previous employers'): {
        'No, at none of my previous employers': 0, "I don't know": 1,
        'Some of my previous employers': 2, 'Yes, at all of my previous employers': 3,
        'Not applicable': -1
    },
    ('Maybe', "No, I don't think it would", 'No, it has not',
     'Yes, I think it would', 'Yes, it has'): {
        "No, I don't think it would": 0, 'No, it has not': 0,   # collapsed — see note above
        'Maybe': 1,
        'Yes, I think it would': 2, 'Yes, it has': 2             # collapsed — see note above
    },
    ('Maybe', "No, I don't think they would", 'No, they do not',
     'Yes, I think they would', 'Yes, they do'): {
        "No, I don't think they would": 0, 'No, they do not': 0,  # collapsed
        'Maybe': 1,
        'Yes, I think they would': 2, 'Yes, they do': 2            # collapsed
    },
    ('Neutral', 'Not applicable to me (I do not have a mental illness)',
     'Not open at all', 'Somewhat not open', 'Somewhat open', 'Very open'): {
        'Not open at all': 0, 'Somewhat not open': 1, 'Neutral': 2,
        'Somewhat open': 3, 'Very open': 4,
        'Not applicable to me (I do not have a mental illness)': -1
    },
    ('Maybe/Not sure', 'No', 'Yes, I experienced', 'Yes, I observed'): {
        'No': 0, 'Maybe/Not sure': 1, 'Yes, I observed': 2, 'Yes, I experienced': 3
    },
    ('Maybe', 'No', 'Not applicable', 'Yes'): {
        'No': 0, 'Maybe': 1, 'Yes': 2, 'Not applicable': -1
    },
    ('No', 'Yes'): {
        'No': 0, 'Yes': 1
    },
    ('Never', 'Not applicable to me', 'Often', 'Rarely', 'Sometimes'): {
        'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Not applicable to me': -1
    },
}

# rebuild vocab_groups fresh here so this cell is self-contained
# and doesn't silently depend on Cell 18 having just run
from collections import defaultdict

vocab_groups = defaultdict(list)
for c in X_cat.columns:
    vocab = tuple(sorted(X_cat[c].dropna().unique(), key=lambda x: str(x)))
    vocab_groups[vocab].append(c)

# apply each map to every column sharing that vocabulary
mapped_cols = []
for vocab, cols in vocab_groups.items():
    if vocab == (np.int64(0), np.int64(1)):
        # q53 — already numeric, nothing to map
        continue
    if vocab not in vocab_maps:
        print(f"WARNING: no map defined for vocabulary {vocab} (columns: {cols})")
        continue
    mapping = vocab_maps[vocab]
    for col in cols:
        X_cat[col] = X_cat[col].map(mapping)
        mapped_cols.append(col)

print(f"Mapped {len(mapped_cols)} columns using {len(vocab_maps)} vocabulary-specific maps")
print(f"Columns mapped: {sorted(mapped_cols)}")

Mapped 35 columns using 21 vocabulary-specific maps
Columns mapped: ['q05', 'q06', 'q07', 'q08', 'q09', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q26', 'q27', 'q28', 'q29', 'q30', 'q31', 'q32', 'q33', 'q34', 'q35', 'q36', 'q37', 'q39', 'q41', 'q42', 'q43', 'q44', 'q45', 'q46', 'q47', 'q48', 'q51', 'q54', 'q55']


In [25]:

# ============================================================
# Cell 19a — Validate the encoding: no leftover strings, no
# unexpected NaNs introduced by an incomplete map.
#
# Widened to catch both the legacy 'object' dtype and pandas'
# newer dedicated 'str' dtype — an earlier version of this check
# only looked for 'object' and missed q06 when it was sitting as
# 'str', silently under-reporting the problem.
# ============================================================
non_numeric = X_cat.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Columns still non-numeric after mapping: {non_numeric}")
print(f"(Expect: empty list — all 36 columns should be numeric)")

null_counts = X_cat.isna().sum()
unexpected_nulls = null_counts[null_counts > 0]
print(f"\nColumns with unexpected NaN after mapping:")
print(unexpected_nulls if len(unexpected_nulls) else "None — all values mapped successfully")

print(f"\nX_cat dtypes summary:")
print(X_cat.dtypes.value_counts())

Columns still non-numeric after mapping: []
(Expect: empty list — all 36 columns should be numeric)

Columns with unexpected NaN after mapping:
None — all values mapped successfully

X_cat dtypes summary:
int64    36
Name: count, dtype: int64


In [26]:
# ============================================================
# Cell 19d — Validate the context frame
# context was rebuilt alongside X_cat in the Cell 16 re-run but
# never independently checked. q25, q56-q58, q62, q63 were not
# part of the q06/q44 investigation, so their missingness status
# is currently unverified.
# ============================================================
print("context missing values by column:")
print(context.isna().sum())

print("\ncontext dtypes:")
print(context.dtypes)

context missing values by column:
q56    0
q57    0
q58    0
q62    0
q63    0
q25    0
dtype: int64

context dtypes:
q56    float64
q57        str
q58        str
q62        str
q63        str
q25      int64
dtype: object


In [27]:
# ============================================================
# Cell 19e — Final spot-check on the columns affected by both
# major bugs in this notebook (Likert corruption + q25 staleness
# indirectly touched this region). Purely confirmatory.
# ============================================================
check_cols = ['q06', 'q30', 'q31', 'q32', 'q33', 'q41', 'q42', 'q53']
for c in check_cols:
    print(f"{c}: {sorted(X_cat[c].dropna().unique())}")

q06: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]
q30: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
q31: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
q32: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]
q33: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]
q41: [np.int64(0), np.int64(1), np.int64(2)]
q42: [np.int64(0), np.int64(1), np.int64(2)]
q53: [np.int64(0), np.int64(1)]


In [28]:

# ============================================================
# Cell 19b — Spot check a few columns post-encoding, including
# the previously-corrupted q30-q33/q41-q42, to confirm clean
# integer output with correct value ranges
# ============================================================
check_cols = ['q30', 'q31', 'q32', 'q33', 'q41', 'q42', 'q53']
for c in check_cols:
    print(f"{c}: {sorted(X_cat[c].dropna().unique())}")

q30: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
q31: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
q32: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]
q33: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]
q41: [np.int64(0), np.int64(1), np.int64(2)]
q42: [np.int64(0), np.int64(1), np.int64(2)]
q53: [np.int64(0), np.int64(1)]


In [29]:

# ============================================================
# Cell 20 — Save processed artifacts for Notebook 3
# This is the genuinely final, fully-encoded feature matrix —
# verified clean of the q25 role bug, the Likert-map string/float
# corruption, and the q06/q44 residual-missingness gap, in that
# order, across this notebook.
# ============================================================
X_cat.to_csv('../data/processed/X_cat.csv', index=False)
context.to_csv('../data/processed/context.csv', index=False)
post_filter_summary.to_csv('../data/processed/post_filter_column_summary.csv')

print("Saved X_cat.csv, context.csv, and post_filter_column_summary.csv to data/processed/")
print(f"\nFinal X_cat shape: {X_cat.shape}")
print(f"Final X_cat dtypes:\n{X_cat.dtypes.value_counts()}")
print(f"\nFinal context shape: {context.shape}")

Saved X_cat.csv, context.csv, and post_filter_column_summary.csv to data/processed/

Final X_cat shape: (1146, 36)
Final X_cat dtypes:
int64    36
Name: count, dtype: int64

Final context shape: (1146, 6)
